# RAQG-QPP: Retrieve Query Variants (BM25, 1-hop) for DL19/DL20

This notebook:
1. Mounts your Google Drive (source of the original queries/qrels).
2. Installs PyTerrier + deps.
3. Clones the `local-qv-generation` branch of your repo.
4. Builds the MS MARCO passage corpus index (one-time, ~20-40 min depending on disk/CPU).
5. Builds the MS MARCO training-query BM25 index (one-time).
6. Runs `retrieve_qvs.py` for `dl_19` and `dl_20` using the queries already embedded in `res/dl_19_bm25.csv` / `res/dl_20_bm25.csv`.
7. Inspects the resulting `qv_res/*.csv` files (left in the local Colab session; copy them to Drive yourself if you want them to persist).

Steps 4-5 only need to run once per Colab **persistent** disk. On a fresh/ephemeral runtime you'll rebuild them each session unless you point `--out_path` at a Drive-backed folder (slower, but persists).

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Install dependencies

In [ ]:
!pip install -q python-terrier pyterrier_dr pyterrier_alpha ir_datasets

# PyTerrier needs a JVM; Colab ships one, but make sure java is on PATH.
!java -version

## 3. Clone the repo

In [ ]:
import os

REPO_DIR = '/content/RAQG_QPP'

if not os.path.exists(REPO_DIR):
    !git clone --branch local-qv-generation https://github.com/Riddhi2587/RAQG_QPP.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git checkout local-qv-generation && git pull

%cd {REPO_DIR}

## 4. Build the MS MARCO passage corpus index (one-time)

Needed for the RBO reranking step (retrieves top-20 docs for the target query and each QV candidate to compare rankings).

This streams the full MS MARCO passage corpus via `ir_datasets` on first run — expect it to take a while and use a few GB of disk under `~/.ir_datasets`.

In [ ]:
DOC_INDEX_PATH = '/content/doc_indices/msmarco-passage.terrier'

if not os.path.exists(DOC_INDEX_PATH):
    !python doc_indices/build_msmarco_passage_index.py --out_path {DOC_INDEX_PATH}
else:
    print('Doc index already exists, skipping build.')

## 5. Build the MS MARCO training-query BM25 index (one-time)

This is the pool that query variants are retrieved *from*. Must be built at `./query_indices/sparse` relative to the repo root, since `query_retrievers.py` hardcodes that path.

In [ ]:
QUERY_INDEX_PATH = os.path.join(REPO_DIR, 'query_indices', 'sparse')

if not os.path.exists(QUERY_INDEX_PATH):
    %cd {REPO_DIR}/query_indices
    !python sparse_indexing_msmarco_only_judged.py
    %cd {REPO_DIR}
else:
    print('Query index already exists, skipping build.')

## 6. Retrieve QVs (BM25, 1-hop) for DL19 and DL20

Uses the queries already embedded in the tracked `res/dl_19_bm25.csv` / `res/dl_20_bm25.csv` runfiles (the 43/54 officially-judged DL19/DL20 queries) as the `--queries_path`, so there's no need to touch the separate queries file on Drive.

In [ ]:
%cd {REPO_DIR}
!python retrieve_qvs.py --dataset_name dl_19 --q_retriever bm25 --hop_num 1 \
    --queries_path ./res/dl_19_bm25.csv \
    --doc_index_path {DOC_INDEX_PATH}

In [ ]:
!python retrieve_qvs.py --dataset_name dl_20 --q_retriever bm25 --hop_num 1 \
    --queries_path ./res/dl_20_bm25.csv \
    --doc_index_path {DOC_INDEX_PATH}

## 7. Inspect and save results

In [ ]:
import pandas as pd

dl19_qvs = pd.read_csv(f'{REPO_DIR}/qv_res/reranked_dl_19_bm25_1hop.csv')
dl20_qvs = pd.read_csv(f'{REPO_DIR}/qv_res/reranked_dl_20_bm25_1hop.csv')

print(dl19_qvs.shape, dl20_qvs.shape)
dl19_qvs.head()